# 01 – Isınma: Aynı soru, iki model

*Hafta 1 ısınma notebook'u.* Aynı soruyu önce bilgisayarınızda çalışan **yerel modele** (Ollama, qwen3.5:4b),
sonra **bulut modele** (Claude) soracağız. Cevapları, süreleri ve harcanan token sayısını karşılaştıracağız.

> Başlamadan önce terminalde `python kontrol.py` çalıştırın; tüm satırlar ✅ (veya ⚠️) olmalı.
>
> Notebook'un sağ üst köşesinde projenin Python ortamı seçili olmalı (Dev Container'da *Python 3*, kendi bilgisayarınızda `.venv`).
> İlk hücrede `ModuleNotFoundError` alırsanız önce buraya bakın.

## Bu notebook neden var?

Kısaca: **modelle ilk kez konuşmak ve proje boyunca kullanacağımız iki modeli tanımak.**

### Ne öğreniyoruz
- LangChain ile bir modelin nasıl çağrıldığını: `model_getir(...)` ile modeli hazırla, `.invoke(...)` ile soruyu sor.
  Yerel model ve Claude için kod **neredeyse aynı**, sadece model değişiyor.
- Modelin cevabının düz bir metin değil, bir **nesne** olduğunu: içinde metnin yanında harcanan token sayısı gibi bilgiler de var.

### Neden iki model
Proje 5 hafta boyunca iki modelle çalışacak ve bir bankada hangisinin seçileceği gerçek bir karar:
- **Yerel model** (Ollama, qwen3.5:4b): ücretsiz, müşteri verisi bilgisayardan dışarı çıkmaz. Ama daha küçük ve daha zayıf bir model.
- **Bulut model** (Claude): daha güçlü. Ama her çağrı ücretli ve soru internet üzerinden dışarı gönderilir.

Bu notebook bu dengeyi ilk kez kendi gözünüzle görmeniz için: aynı soruya iki modelin verdiği cevabı, süreyi ve token sayısını yan yana karşılaştıracaksınız.

### Neden 5 hafta boyunca önemli
- **Görev 5**'te aynı karşılaştırmayı 30 talep üzerinde, ölçerek yapacaksınız (`python eval/calistir.py --model bulut`).
- İleriki haftalarda müşteri verisini modele göndermeden önce maskelemeyi göreceğiz. Bulut modelin "veri dışarı çıkar" tarafının çözümü bu.
- **Token** sayısı maliyetin ölçüsü. Hafta 5'teki maliyet raporunda bu sayılara geri döneceğiz.

### Dikkat edeceğiniz şey
Modele hiçbir bağlam vermeden soruyoruz: kim olduğunu, hangi bankada çalıştığını söylemiyoruz.
Cevapları okurken modelin **ne bildiğine ve neyi bilmediğine** dikkat edin.

> **Tek cümleyle:** Bu notebook modelle nasıl konuşulacağını gösterir, `02_yapilandirilmis_cikti` ise modelin cevabının bir programın parçası olarak nasıl kullanılacağını.

In [8]:
# Proje kökünü bul ve src/ klasörünü import yoluna ekle
import sys
from pathlib import Path

PROJE_KOKU = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kontrol.py").exists())
sys.path.insert(0, str(PROJE_KOKU / "src"))

from asistan.ayarlar import ayar, model_getir

print("Yerel model:", ayar("OLLAMA_MODEL"), "@", ayar("OLLAMA_URL"))

Yerel model: qwen3.5:4b @ http://localhost:11434


### Satır satır açıklama

**`# Proje kökünü bul ve src/ klasörünü import yoluna ekle`**
`#` ile başlayan satır **yorumdur**. Python bu satırı çalıştırmaz, sadece okuyana bilgi verir.

**`import sys`**
Python'la birlikte gelen `sys` modülünü yükler (ayrıca kurmak gerekmez). Bu modülle Python'un kendi ayarlarına erişiriz. Bize lazım olan `sys.path` listesi: Python, `import` gördüğünde modülleri bu listedeki klasörlerde arar.

**`from pathlib import Path`**
`pathlib` modülünden `Path` aracını alır. `Path` dosya ve klasör yollarını metin olarak değil, **nesne** olarak tutar:
- Yolları `/` işaretiyle birleştirebiliriz: `klasor / "dosya.py"`.
- Windows'taki `\` ile Mac'teki `/` farkını kendisi halleder.

**`PROJE_KOKU = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kontrol.py").exists())`**
En karmaşık satır bu. İçten dışa doğru:

| Parça | Ne yapar | Örnek değer |
|---|---|---|
| `Path.cwd()` | Şu an çalışılan klasör (*current working directory*). Notebook kendi klasöründe çalışır. | `.../softtech_agentic/haftalar/hafta-1` |
| `Path.cwd().parents` | Bu klasörün bütün üst klasörleri, yakından uzağa | `.../haftalar`, `.../softtech_agentic`, `.../Desktop`, ... `/` |
| `[Path.cwd(), *Path.cwd().parents]` | Hepsini tek listede toplar. `*` işareti "`parents` içindekileri tek tek listeye dök" demek. | `[hafta-1, haftalar, softtech_agentic, Desktop, ...]` |
| `p / "kontrol.py"` | Listedeki her klasörün (`p`) sonuna `kontrol.py` ekler | `.../hafta-1/kontrol.py` |
| `.exists()` | O dosya gerçekten var mı? `True` / `False` | |
| `p for p in [...] if ...` | Klasörleri sırayla dolaşır, sadece içinde `kontrol.py` olanları seçer | |
| `next(...)` | Seçilenlerden **ilkini** alır ve aramayı bitirir | |

Arama şöyle ilerler:
- `hafta-1/kontrol.py` var mı? **Yok**, geç.
- `haftalar/kontrol.py` var mı? **Yok**, geç.
- `softtech_agentic/kontrol.py` var mı? **Var**, bunu seç.

Sonuç `PROJE_KOKU` değişkenine kaydedilir: `.../softtech_agentic`

*Neden bu kadar uğraşıyoruz?* Proje Masaüstünde, başka bir sürücüde ya da Dev Container'da (`/workspace`) olabilir. Yolu koda elle yazmak yerine `kontrol.py` dosyasını "işaret taşı" olarak kullanıyoruz. Böylece kod her bilgisayarda çalışır.

**`sys.path.insert(0, str(PROJE_KOKU / "src"))`**
- `PROJE_KOKU / "src"` → proje kökünün sonuna `src` ekler: `.../softtech_agentic/src`
- `str(...)` → yolu düz metne çevirir. `sys.path` metinlerden oluşan bir liste olduğu için gerekli.
- `sys.path.insert(0, ...)` → bu yolu listenin **0. sırasına**, yani en başına ekler.

*Neden?* `asistan` klasörü `src/` içinde duruyor ve Python normalde oraya bakmaz. `src/` yolunu ekleyince `import asistan` çalışır. En başa eklememizin sebebi, aynı adlı başka bir paket varsa önce bizimkinin bulunması.

**`from asistan.ayarlar import ayar, model_getir`**
- `asistan.ayarlar` → `src/asistan/ayarlar.py` dosyası. Nokta, klasör ayracı gibi düşünülebilir.
- `import ayar, model_getir` → o dosyadaki iki fonksiyonu bu notebook'ta kullanılabilir yapar.

Bu satır çalışınca arka planda şunlar olur:
1. Python önce `src/asistan/__init__.py` dosyasını çalıştırır. Bir klasörden ilk kez bir şey alındığında hep bu dosya çalışır.
2. `__init__.py` de `asistan.py` dosyasını yükler. O dosya `pydantic` paketini ister. Notebook yanlış ortamda (kernel) açıksa `ModuleNotFoundError` hatası **tam burada** çıkar. Çözümü: sağ üstten `.venv` kernel'ini seçmek.
3. `ayarlar.py` çalışır ve `.env` dosyasını okur.

Alınan fonksiyonlar:
- `ayar("OLLAMA_MODEL")` → `.env` dosyasındaki değeri döner. Değer boşsa anlaşılır bir hata verir.
- `model_getir("yerel")` → kullanıma hazır bir model nesnesi döner (2. ve 3. bölümde kullanılıyor).

**`print("Yerel model:", ayar("OLLAMA_MODEL"), "@", ayar("OLLAMA_URL"))`**
- `ayar("OLLAMA_MODEL")` → model adı: `qwen3.5:4b`
- `ayar("OLLAMA_URL")` → Ollama'nın adresi: `http://localhost:11434`
- `print(...)` → virgülle ayrılan parçaları aralarına boşluk koyarak yan yana yazar.

Çıktı:
```
Yerel model: qwen3.5:4b @ http://localhost:11434
```
Amaç kontrol: ayarlar doğru okunuyor mu, hangi modelle hangi adrese bağlanacağız? `.env` eksikse hata erkenden, burada ortaya çıkar.

> **Özet:** Proje klasörünü bul, `src/` klasörünü Python'a göster, ayar fonksiyonlarını yükle ve ayarların doğru olduğunu ekranda gör.

## 1. Soru

İki modele de **aynı soruyu** soracağız. Böylece cevaplardaki fark sadece modelden kaynaklanır.
Kendi sorunuzu denemek isterseniz sadece bu satırı değiştirip notebook'u baştan çalıştırın.

In [9]:
SORU = "Kartımı kaybettim, ne yapmalıyım?"

### Satır satır açıklama

**`SORU = "Kartımı kaybettim, ne yapmalıyım?"`**
- Soruyu `SORU` adlı bir değişkene kaydeder. Sonraki hücrelerde iki model de bu değişkeni kullanır.
- Değişken adının **büyük harfle** yazılması Python'da bir alışkanlıktır: "bu değer notebook boyunca sabit, değiştirilmeyecek" demektir.
- Soruyu tek bir yerde tutmanın faydası: değiştirmek istediğinizde iki yerde değil, tek yerde değiştirirsiniz. İki modele farklı soru sorma hatası da yapılamaz.

## 2. Yerel model (Ollama)

Model bilgisayarınızda çalışır: ücretsizdir ve soru bilgisayarınızdan dışarı çıkmaz.

In [10]:
import time

yerel_model = model_getir("yerel")

baslangic = time.perf_counter()
yerel_cevap = yerel_model.invoke(SORU)
yerel_sure = time.perf_counter() - baslangic

print(f"Süre: {yerel_sure:.1f} sn\n")
print(yerel_cevap.text)

Süre: 30.5 sn

Kartınızı kaybettiğinizde panik yapmaktan kaçınmanız ve **hızlıca** aşağıdaki adımları izlemeniz gerekir. Kartınızın çalınması durumunda en önemli şey, kötüye kullanılabileceğini önlemektir.

İzlemeniz gereken acil adımlar şunlardır:

### 1. En Önemli Adım: Bankanızla İletişime Geçin
Kartınızı bulamadığınız an, **hemen** bankanızın müşteri hizmetlerini arayarak kartı "kayıp" olarak bildiriniz.
*   Kart numarasını ve son 4 hanesini hatırlıyorsanız, doğrudan bankanın acil iletişim hattını arayın (genellikle 0850 veya 0800 numaralarıdır).
*   Hatırlamıyorsanız, bankanızın web sitesinden "Kayıp Kart Bildirimi" seçeneğini kullanarak online başvuru yapabilirsiniz.
*   **Neden bu ilk adım?** Bu işlemi yaptığınız anda kart üzerindeki tüm cüzdanların (online alışveriş, bakiye çekme vb.) bloke olması sağlanır ve para kaybı önlenir.

### 2. Kartınızı İade Edin
Bankanız ile iletişime geçtiğinizden sonra size bir **kayıp kart numarası** verecektir. Bu yeni numara ile bankanızın web s

### Satır satır açıklama

**`import time`**
Python'la gelen `time` modülünü yükler. Süre ölçmek için kullanacağız.

**`yerel_model = model_getir("yerel")`**
- `model_getir` ilk hücrede `ayarlar.py` dosyasından alınmıştı. `"yerel"` → Ollama'daki qwen3.5:4b modeli.
- Fonksiyon bir **model nesnesi** döner (LangChain'in `ChatOllama` sınıfı) ve `yerel_model` değişkenine kaydedilir.
- Bu satır modele **henüz hiçbir şey sormaz**, sadece bağlantı ayarlarını hazırlar. Bu yüzden çok hızlıdır.

**`baslangic = time.perf_counter()`**
- `time.perf_counter()` → bilgisayarın hassas saatinden o anki değeri (saniye cinsinden) okur.
- Bu değer tek başına bir anlam taşımaz. Önemli olan, iki ölçüm arasındaki **fark**.
- Soruyu sormadan hemen önce saati `baslangic` değişkenine kaydediyoruz.

**`yerel_cevap = yerel_model.invoke(SORU)`**
- `.invoke(...)` modeli **çalıştırır**: soruyu gönderir ve cevap gelene kadar bekler. Hücrenin en uzun süren satırı bu.
- LangChain'de neredeyse her şey `.invoke()` ile çalıştırılır: modeller, zincirler ve ileriki haftalarda ajanlar.
- Gelen cevap bir `AIMessage` (yapay zekâ mesajı) nesnesidir ve `yerel_cevap` değişkenine kaydedilir. İçinde metnin yanında harcanan token sayısı gibi başka bilgiler de var. Bunları 4. bölümde kullanacağız.

**`yerel_sure = time.perf_counter() - baslangic`**
- Cevap geldikten sonra saati tekrar okuyup başlangıç değerini çıkarıyoruz.
- Sonuç, cevabın kaç saniyede geldiği. `yerel_sure` değişkenine kaydedilir.

**`print(f"Süre: {yerel_sure:.1f} sn\n")`**
- `f"..."` → **f-string**. Süslü parantez `{...}` ile metnin içine değişken yerleştirilir.
- `:.1f` → sayıyı virgülden sonra **1 basamakla** yaz. Bu olmasa `26.31849...` gibi uzun bir sayı çıkar.
- `\n` → satır sonu. Süre ile cevap arasında boş bir satır bırakır.

**`print(yerel_cevap.text)`**
`.text` cevabın **düz metin** halini verir.

Örnek çıktı (kısaltılmış):
```
Süre: 26.3 sn

Kartınızı kaybettiğinizde panik yapmaktan kaçınmanız ve **hızlıca** aşağıdaki adımları izlemeniz gerekir.
### 1. En Önemli Adım: Bankanızı Bilgilendirin
...
```

> **Dikkat edin:**
> - **Süre** bilgisayarınızın gücüne bağlıdır. İlk çalıştırmada model belleğe yükleneceği için daha uzun sürebilir. Hücreyi ikinci kez çalıştırıp farkı görün.
> - **Cevabı dikkatle okuyun.** Bir denemede model, "bankanız size bir *kayıp kart numarası* verecek, bununla kartı *iade edin*" gibi **gerçekte olmayan adımlar** yazdı. Küçük modeller kulağa inandırıcı gelen ama yanlış bilgiler üretebilir. Buna *halüsinasyon* denir.

## 3. Bulut model (Claude)

Model Anthropic'in sunucularında çalışır: daha güçlüdür, ama ücretlidir ve soru internet üzerinden dışarı gönderilir.
Bu hücre için `.env` dosyasında `ANTHROPIC_API_KEY` tanımlı olmalı.

In [11]:
import os

if os.getenv("ANTHROPIC_API_KEY"):
    bulut_model = model_getir("bulut")

    baslangic = time.perf_counter()
    bulut_cevap = bulut_model.invoke(SORU)
    bulut_sure = time.perf_counter() - baslangic

    print(f"Süre: {bulut_sure:.1f} sn\n")
    print(bulut_cevap.text)
else:
    bulut_cevap = None
    print("ANTHROPIC_API_KEY yok, bu hücre atlandı. Anahtarı .env dosyasına ekleyip tekrar deneyin.")

Süre: 9.2 sn

Hangi kartı kaybettiğinizi belirtmediniz. En yaygın durumlar için yapmanız gerekenler şöyle:

## 💳 Banka veya kredi kartı
1. **Hemen bankanızı arayın** ve kartı iptal ettirin veya geçici olarak bloke edin. Çoğu banka 7/24 hizmet verir. Kartın arkasındaki veya bankanızın web sitesindeki numarayı kullanabilirsiniz.
2. **Mobil bankacılık veya internet bankacılığından** da kartı kapatabilirsiniz. Çoğu bankada "Kartı geçici kapat" veya "Kayıp/çalıntı bildir" seçeneği vardır.
3. **Son işlemlerinizi kontrol edin.** Size ait olmayan bir harcama görürseniz bankaya itiraz edin.
4. **Yeni kart talep edin.**
5. Kartınız çalındıysa veya yetkisiz işlem olduysa **polise bildirim** yapmanız da faydalı olur.
6. Kartı kayıtlı olduğu yerlerde (online alışveriş siteleri, otomatik ödemeler, abonelikler) güncelleyin.

## 🪪 Kimlik kartı (T.C. kimlik)
1. **Nüfus müdürlüğüne** başvurun. Randevuyu e-Devlet veya Nüfus ve Vatandaşlık İşleri (NVİ) üzerinden alabilirsiniz.
2. Kimliğiniz kötü niyetle k

### Satır satır açıklama

**`import os`**
Python'la gelen `os` (*operating system*, işletim sistemi) modülünü yükler. **Ortam değişkenlerini** okumak için kullanıyoruz.

**`if os.getenv("ANTHROPIC_API_KEY"):`**
- `os.getenv(...)` → bu adlı ortam değişkeninin değerini okur. Değer `.env` dosyasından gelir: ilk hücrede `ayarlar.py` yüklenirken `.env` içindeki her şey ortam değişkenlerine aktarılmıştı. Değişken yoksa `None` döner.
- `if ...:` → koşul doğruysa altındaki girintili bloğu çalıştırır. Python'da dolu bir metin "doğru", boş metin veya `None` "yanlış" sayılır. Yani bu satır "anahtar varsa" demek.
- *Neden kontrol ediyoruz?* Herkesin Claude anahtarı olmayabilir. Anahtar yoksa hücre hata verip durmak yerine nazikçe atlanır.

**`bulut_model = model_getir("bulut")`**
Bu sefer `"bulut"` istiyoruz: LangChain'in `ChatAnthropic` sınıfı. Model adı koda yazılmaz, `.env` dosyasındaki `CLAUDE_MODEL` değerinden okunur.

**`baslangic = time.perf_counter()` … `bulut_sure = time.perf_counter() - baslangic`**
Yerel modeldeki ölçümün aynısı: önce saati oku, **aynı soruyu** sor, sonra geçen süreyi hesapla. Kod neredeyse birebir aynı, değişen sadece model. LangChain'in faydası bu: farklı şirketlerin modelleri aynı `.invoke()` komutuyla çağrılır.

**`print(f"Süre: {bulut_sure:.1f} sn\n")` ve `print(bulut_cevap.text)`**
Süreyi ve cevabın düz metnini yazdırır.

*Neden `.content` değil de `.text`?* Claude bazen cevap vermeden önce kendi içinde bir **"düşünme"** adımı yapar. Bu durumda `.content` düz metin değil, bir blok listesi olur:
```python
[{"type": "thinking", ...}, {"type": "text", "text": "Kartınızı kaybettiyseniz..."}]
```
`.text` bu listeden sadece metin kısmını alır. Yerel modelde de aynı şekilde çalıştığı için iki modelde de `.text` kullanıyoruz.

**`else:`**
"Koşul doğru değilse", yani anahtar yoksa altındaki bloğu çalıştır. `if` ile aynı girinti hizasında olmalı.

**`bulut_cevap = None`**
Değişkene "boş" değeri verir. Bir sonraki hücre bu değişkene bakarak Claude'un çalışıp çalışmadığını anlayacak. Bu satır olmasa, anahtarı olmayan katılımcıda bir sonraki hücre `NameError` hatası verir.

**`print("ANTHROPIC_API_KEY yok, bu hücre atlandı. ...")`**
Bilgi mesajı yazar. Hata vermez, notebook çalışmaya devam eder.

Örnek çıktı (kısaltılmış):
```
Süre: 9.2 sn

Kartınızı kaybettiyseniz hızlı davranmanız önemli. Banka veya kredi kartı için şunları yapın:
## Banka / Kredi Kartı
1. **Kartı hemen bloke edin.**
...
## Kimlik Kartı (T.C. Kimlik Kartı)
...
```

> **Dikkat edin:** Claude "kart" kelimesinin banka kartı mı, kimlik kartı mı olduğunu bilmediği için birkaç ihtimali birden cevaplıyor. Modele bir banka asistanı olduğunu söylemedik. Ona bu bağlamı vermenin yolu **sistem mesajıdır**, `src/asistan/tek_cagri.py` dosyasında bunu göreceksiniz.

## 4. Karşılaştırma

İki modelin süresini, harcadığı token sayısını ve cevabın uzunluğunu yan yana görelim.
**Token**, modelin metni işlerken kullandığı küçük parçalardır (yaklaşık bir hece ya da kısa bir kelime). Bulut modellerde ücret token sayısına göre ödenir.

In [12]:
sonuclar = [("Yerel", yerel_cevap, yerel_sure)]
if bulut_cevap is not None:
    sonuclar.append(("Claude", bulut_cevap, bulut_sure))

print(f"{'Model':<8} {'Süre (sn)':>10} {'Girdi token':>12} {'Çıktı token':>12} {'Karakter':>9}")
for ad, cevap, sure in sonuclar:
    token = cevap.usage_metadata
    print(f"{ad:<8} {sure:>10.1f} {token['input_tokens']:>12} {token['output_tokens']:>12} {len(cevap.text):>9}")

Model     Süre (sn)  Girdi token  Çıktı token  Karakter
Yerel          30.5           25          644      2145
Claude          9.2           29          969      1580


### Satır satır açıklama

**`sonuclar = [("Yerel", yerel_cevap, yerel_sure)]`**
- Bir **liste** oluşturur. Listenin tek elemanı üç bilgiden oluşan bir grup: model adı, cevap ve süre. Python'da bu gruba *tuple* denir.
- Yerel model her zaman çalıştığı için listeye baştan ekleniyor.

**`if bulut_cevap is not None:`**
- `is not None` → "boş değilse". Yani Claude hücresi gerçekten çalıştıysa.
- Bir önceki hücrede anahtar yoksa `bulut_cevap = None` yapmıştık. Bu satır o durumu yakalar.

**`sonuclar.append(("Claude", bulut_cevap, bulut_sure))`**
`.append(...)` listenin sonuna yeni bir eleman ekler. Artık listede iki model var.

**`print(f"{'Model':<8} {'Süre (sn)':>10} {'Girdi token':>12} {'Çıktı token':>12} {'Karakter':>9}")`**
Tablonun **başlık satırını** yazar. Süslü parantezlerin içinde değişken değil, düz metin var. Amaç, metne genişlik vermek:
- `:<8` → 8 karakterlik alana **sola** yasla
- `:>10` → 10 karakterlik alana **sağa** yasla (sayılar sağa yaslı daha kolay okunur)

Böylece başlıklar ve altındaki sayılar aynı hizada durur.

**`for ad, cevap, sure in sonuclar:`**
- Listedeki her grubu sırayla dolaşır.
- `ad, cevap, sure` → grubun üç parçasını tek seferde üç ayrı değişkene açar. Buna *unpacking* (açma) denir.

**`token = cevap.usage_metadata`**
`usage_metadata` → cevap nesnesinin içindeki **kullanım bilgisi**. Bir sözlüktür:
```python
{"input_tokens": 25, "output_tokens": 874, "total_tokens": 899}
```
- `input_tokens` → soruyu okumak için harcanan token (girdi)
- `output_tokens` → cevabı yazmak için harcanan token (çıktı)

**`print(f"{ad:<8} {sure:>10.1f} {token['input_tokens']:>12} {token['output_tokens']:>12} {len(cevap.text):>9}")`**
Her model için bir satır yazar. Genişlikler başlık satırıyla aynı:
- `{sure:>10.1f}` → süre, sağa yaslı, virgülden sonra 1 basamak
- `{token['input_tokens']:>12}` → sözlükten girdi token sayısı. Köşeli parantezde **tek tırnak** var, çünkü dışarıda f-string'in çift tırnağı kullanılıyor.
- `{len(cevap.text):>9}` → `len(...)` cevap metninin kaç karakter olduğunu verir

Örnek çıktı:
```
Model     Süre (sn)  Girdi token  Çıktı token  Karakter
Yerel          26.3           25          874      2759
Claude          9.2           29          896      1469
```

> **Tabloyu okuyun:**
> - **Süre:** Bu denemede Claude yaklaşık 3 kat daha hızlı. Yerel modelin hızı bilgisayarınıza, Claude'unki internet bağlantısına ve sunucunun yoğunluğuna bağlıdır.
> - **Girdi token:** Aynı soru iki modelde farklı sayıda token tutuyor (25 ve 29). Her model metni kendi yöntemiyle parçalara ayırır.
> - **Çıktı token ve karakter:** Claude'un cevabı daha kısa (1469 karakter), ama çıktı token'ı yerel modelle neredeyse aynı (896). Fark, Claude'un cevaptan önce yaptığı ve bizim görmediğimiz **"düşünme"** adımından geliyor. O kısım da çıktı token'ına, yani **faturaya** dahil.

## Tartışma

Aşağıdaki soruları düşünün. Cevaplarınızı bu hücrenin altına yeni bir markdown hücresi ekleyerek yazın (Görev 1 teslimi).

1. **Hız:** Hangi model daha hızlı cevap verdi? Hücreleri birkaç kez çalıştırın: süreler değişiyor mu? Değişiyorsa neden?
2. **Kalite:** Hangi cevabı olduğu gibi bir banka müşterisine gönderirdiniz? Cevaplarda yanlış ya da uydurma bir bilgi var mı?
3. **Bağlam:** İki model de Örnek Bank'ın kayıp kart prosedürünü biliyor mu? Bilmiyorsa bu bilgiyi modele nasıl verebiliriz?
4. **Maliyet ve gizlilik:** Soruda müşterinin adı ve kart numarası olsaydı, hangi modele göndermek risk olurdu? Günde 10.000 talep gelen bir bankada hangi modeli, neden seçerdiniz?